# Fraud detection with a small random forest
Synthetic transactions only. The two CSV cohorts use different fraud-label rules.
Train on the active CSV, then reuse `predict` with the saved model.
Seeds record intended controls; they do not guarantee reproducibility across environments.

In [ ]:
from pathlib import Path
from typing import TypedDict

from features import _feature_row
from sklearn.metrics import precision_score, recall_score, roc_auc_score

n_estimators = 24
max_depth = 5


class Transaction(TypedDict):
    amount: float
    transaction_velocity: int
    foreign_transaction: bool
    account_age_days: int


def train():
    import joblib
    import numpy as np
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split

    frame = pd.read_csv("data/transactions.csv")
    columns = [
        "amount",
        "transaction_velocity",
        "foreign_transaction",
        "account_age_days",
    ]
    x_train, x_test, y_train, y_test = train_test_split(
        frame[columns].to_numpy(),
        frame["fraud"].to_numpy(),
        test_size=0.25,
        random_state=42,
        stratify=frame["fraud"],
    )
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=1,
    )
    model.fit(x_train, y_train)
    Path("artifacts").mkdir(exist_ok=True)
    joblib.dump(model, "artifacts/model.joblib", compress=3)
    Path("artifacts/trained.txt").write_text("Training completed.\n")
    return np.asarray(y_test), model.predict_proba(x_test)[:, 1]


def predict(transaction: Transaction) -> dict[str, list[tuple[int, float]]]:
    """Score one transaction using the saved forest, without training on import."""
    import joblib

    model = joblib.load(Path(__file__).parent / "artifacts/model.joblib")
    row = _feature_row(
        transaction["amount"],
        transaction["transaction_velocity"],
        transaction["foreign_transaction"],
        transaction["account_age_days"],
    )
    probability = float(model.predict_proba([row])[0, 1])
    return {"predictions": [(int(probability >= 0.5), probability)]}


if __name__ == "__main__":
    truth, scores = train()
    predicted = [int(score >= 0.5) for score in scores]
    roc_auc = roc_auc_score(truth, scores)
    precision = precision_score(truth, predicted, zero_division=0)
    recall = recall_score(truth, predicted, zero_division=0)